In [1]:
# Same as ver03 in terms of codes, but the whole data is updated

In [2]:
# Repository setup and shared helper code
import os
import sys
import runpy
from pathlib import Path

import joblib
import keras
import math
import numpy as np
import pandas as pd

possible_roots = [Path.cwd(), *Path.cwd().parents]
root_path = next((base.resolve() for base in possible_roots if (base / "src").exists()), None)
if root_path is None:
    raise FileNotFoundError("Could not locate the repository root containing src/")
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

from src.paths import MGT_MODEL_DIR, OFFSHORE_MICROGRID_DATA_DIR, output_path, require_path

src_dir = root_path / "src"
for source_file in [
    "plotting.py",
    "thermodynamic_model_functions.py",
    "design_data.py",
    "components_definition.py",
    "components_coefficients.py",
    "experimental_data_post_process_functions.py",
    "experimental_data_post_process_functions_psi_data.py",
    "functions_for_offshore_microgrid_optimization_im_gf_ver02.py",
]:
    code = (src_dir / source_file).read_text(encoding="utf-8")
    exec(compile(code, str(src_dir / source_file), "exec"), globals())


In [3]:
name = "Offshore Microgrid Optimization - IM - GF - Day 1"

main_path = str(output_path("models", name))
path_training_history = str(output_path("models", name, "Training History"))
path_trained_model_and_scaler = str(output_path("models", name, "Trained Model"))
path_saving_figure = str(output_path("figures", name))
path_saving_data_results = str(output_path("results", name))


In [4]:
# Load required week and operation data from repository artifacts
week_data_path = require_path(OFFSHORE_MICROGRID_DATA_DIR / "integrated_data_complete.xlsx")
week_data = pd.read_excel(week_data_path, sheet_name="week 1")

GFA_operation_df_path = require_path(OFFSHORE_MICROGRID_DATA_DIR / "GFA_operation_df.pkl")
GFA_operation_df = pd.read_pickle(GFA_operation_df_path)

GFC_operation_df_path = require_path(OFFSHORE_MICROGRID_DATA_DIR / "GFC_operation_df.pkl")
GFC_operation_df = pd.read_pickle(GFC_operation_df_path)

globals().update(runpy.run_path(str(src_dir / "offshore_microgrid_optimization_gfs_data_generation.py"), init_globals=globals()))


2026-05-29 11:11:50.489823: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
week_data['P_net [MW]'] = week_data['P_prod_WT [MW]']-(week_data['P_dem_GFA [MW]']+week_data['P_dem_GFB [MW]']+week_data['P_dem_GFC [MW]'])


plot_params = ['P_prod_WT [MW]', 'P_dem_GFA [MW]', 'Q_dem_GFA [MW]', 'P_dem_GFB [MW]',
       'Q_dem_GFB [MW]', 'P_dem_GFC [MW]', 'Q_dem_GFC [MW]','P_net [MW]']

plot_params = ['P_net [MW]']

for day in range(7):
    str_row = day*24
    end_row = (day+1)*24
    print('Day:',day)
    for param in plot_params:
        simple_plotter(np.arange(24), [week_data[param].iloc[str_row:end_row]],y_label=param)





Day: 0
Day: 1
Day: 2
Day: 3


Day: 4
Day: 5
Day: 6


In [6]:
GFC_operation_df = GFC_operation_df[(GFC_operation_df['T_amb_GFC [degC]']<week_data['T_amb [degC]'].max()+1)&
                (GFC_operation_df['T_amb_GFC [degC]']>week_data['T_amb [degC]'].min()-1)]


GFA_operation_df = GFA_operation_df[(GFA_operation_df['T_amb_GFA [degC]']<week_data['T_amb [degC]'].max()+1)&
                (GFA_operation_df['T_amb_GFA [degC]']>week_data['T_amb [degC]'].min()-1)]


In [7]:
# Load MGT model artifacts from repository artifacts
scaler_MGT = joblib.load(require_path(MGT_MODEL_DIR / "model1_scaler.save"))
model_MGT = keras.models.load_model(require_path(MGT_MODEL_DIR / "model1.h5"))
input_parameters_MGT = ["P_ref [kW]", "T1 [degC]"]
output_parameters_MGT = ["m_f [gr/s]", "TOT [degC]", "N_act [rpm]", "P_act [kW]"]


In [8]:
# resizing the elements so we have a conceptual MG with good sizing
week_data_reserve = week_data.copy()
df = week_data_reserve.copy()


week_data_reserve = week_data.copy()
df = week_data_reserve.copy()



In [9]:
row_begin_day_1 = 0 * 24
row_end_day_1 = row_begin_day_1 + 24
df_study_day_1 = df.iloc[row_begin_day_1:row_end_day_1].copy()

row_begin_day_2 = 1 * 24
row_end_day_2 = row_begin_day_2 + 24
df_study_day_2 = df.iloc[row_begin_day_2:row_end_day_2].copy()

row_begin_day_3 = 2 * 24
row_end_day_3 = row_begin_day_3 + 24
df_study_day_3 = df.iloc[row_begin_day_3:row_end_day_3].copy()

row_begin_day_4 = 3 * 24
row_end_day_4 = row_begin_day_4 + 24
df_study_day_4 = df.iloc[row_begin_day_4:row_end_day_4].copy()

row_begin_day_5 = 4 * 24
row_end_day_5 = row_begin_day_5 + 24
df_study_day_5 = df.iloc[row_begin_day_5:row_end_day_5].copy()

row_begin_day_6 = 5 * 24
row_end_day_6 = row_begin_day_6 + 24
df_study_day_6 = df.iloc[row_begin_day_6:row_end_day_6].copy()

row_begin_day_7 = 6 * 24
row_end_day_7 = row_begin_day_7 + 24
df_study_day_7 = df.iloc[row_begin_day_7:row_end_day_7].copy()

## Optimization

In [10]:
df = df_study_day_1

In [11]:
observation_window = 6
delta_t_time_step = 3600

observation_window_daily = 24

In [12]:
# optimization parameters with [min, max, initial guess]


opt_param_dict = {
'P_GFA_to_GFC':[-20,20,10],
'P_GFA_to_ELZ':[0,100,0],
'FR_GFA_GT1':[0.516,1,1],
# 'FR_GFA_GT2':[0,1,1],
# 'FR_GFA_GT3':[0,1,1],
# 'FR_GFA_GT4':[0.516,1,1],
}

x_band, x_band_statements, parameter_definition, x_initial, no_initial_pop, initial_pop = MG_parameter_boundary_def(opt_param_dict, observation_window)



In [13]:
17*6

102

In [14]:
sell=False
buy=False
hours=6
day_hours = 24

In [15]:
reserve_H2_cb = 0* np.ones(round(7*24/hours)+1)
reserve_H2_op = 0* np.ones(round(7*24/hours)+1)


In [16]:
display_step=1000

In [17]:
period = 1

reserve_H2_cb_per1 = reserve_H2_cb[period-1]
reserve_H2_op_per1 = reserve_H2_op[period-1]


MG_run_cb_per1, cost_total_cb_per1, penalty_total_cb_per1 = MG_run_cb(df.iloc[(period-1)*hours:(period)*hours], reserve_H2_cb[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',MG_run_cb_per1['cost_total [EUR]'].sum())
x_initial_from_cb = extract_x_span_for_initialization(MG_run_cb_per1, x_band_statements)    
simulation_per1, x_record_per1, error_record_per1 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, x_initial_from_cb, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='Nelder-Mead',sell=sell,buy=buy,display_step=display_step)



Total cost: 40132.59267495275


 
>>>>>> itr =  0  cost =  6493.696
Total cost: 6860.737
Penalty cost: 0.0
NOx emission cost: 472.243
H2 incentive: 377.041
Reserved H2 for the next round: 0.917
Deviations cost: 10
Operation cost: 41164.421
>>>>>>>>>>>>> The optimization is finished.


In [18]:
opt_solution_index_per1 = np.where(error_record_per1 == error_record_per1.min())
opt_solution_index_per1 = opt_solution_index_per1[0][0]
best_simulation_per1 = simulation_per1[opt_solution_index_per1]
best_itr_x_per1 = x_record_per1[opt_solution_index_per1]

display(best_itr_x_per1)

array([ 5.00000000e-01,  8.35060375e-01,  1.00000000e+00,  5.00000000e-01,
        5.88190679e-01, -2.41528138e-04,  5.00000000e-01,  7.23842070e-01,
       -2.41528138e-04,  5.00000000e-01,  7.22283940e-01, -2.41528138e-04,
        5.00000000e-01,  8.22131874e-01, -2.41528138e-04,  5.00000000e-01,
        8.76893458e-01, -2.41528138e-04])

In [19]:
best_simulation_per1

,T_amb [degC],P_dem_GFA [MW],Q_dem_GFA [MW],P_dem_GFB [MW],Q_dem_GFB [MW],P_dem_GFC [MW],Q_dem_GFC [MW],P_prod_WT [MW],NG_Price [EUR/kg],P_GFA_GT1 [MW],P_GFA_GT2 [MW],P_GFA_GT3 [MW],P_GFA_GT4 [MW],P_GFC_GT1 [MW],P_GFC_GT2 [MW],P_GFC_GT3 [MW],FR_GFA_GT1 [-],FR_GFA_GT2 [-],FR_GFA_GT3 [-],FR_GFA_GT4 [-],FR_GFC_GT1 [-],FR_GFC_GT2 [-],FR_GFC_GT3 [-],m_NG_GFA_GT1 [kg/s],m_H2_GFA_GT1 [kg/s],m_NG_GFA_GT2 [kg/s],m_H2_GFA_GT2 [kg/s],m_NG_GFA_GT3 [kg/s],m_H2_GFA_GT3 [kg/s],m_NG_GFA_GT4 [kg/s],m_H2_GFA_GT4 [kg/s],m_NG_GFC_GT1 [kg/s],m_H2_GFC_GT1 [kg/s],m_NG_GFC_GT2 [kg/s],m_H2_GFC_GT2 [kg/s],m_NG_GFC_GT3 [kg/s],m_H2_GFC_GT3 [kg/s],eff_GFA_GT1 [-],eff_GFA_GT2 [-],eff_GFA_GT3 [-],eff_GFA_GT4 [-],eff_GFC_GT1 [-],eff_GFC_GT2 [-],eff_GFC_GT3 [-],P_imbalance_GFA [MW],P_imbalance_GFB [MW],P_imbalance_GFC [MW],P_GFA_to_GFB [MW],P_GFB_from_GFA [MW],P_GFA_to_GFC [MW],P_GFA_from_GFC [MW],P_GFC_from_GFA [MW],P_ELH_GFA [MW],P_ELH_GFB [MW],P_ELH_GFC [MW],P_GFA_to_ELZ [MW],cost_NG [EUR],cost_NG_tax [EUR],cost_NOx_tax [EUR],cost_MNT_GT_total [EUR],cost_total [EUR],penalty_H2 [EUR],penalty_P_imbalance [EUR],penalty_total [EUR],H2_production [kg/s],H2_consumption [kg/s],H2_tank_aval [kg/s],H2_tank_var [kg/s],P_GFA_GT1_dev [-],P_GFA_GT2_dev [-],P_GFA_GT3_dev [-],P_GFA_GT4_dev [-],P_GFC_GT1_dev [-],P_GFC_GT2_dev [-],P_GFC_GT3_dev [-],P_imbalance [MW]
0,8.116876,30.692966,10.202288,6.694094,2.168736,48.769910,7.333055,115.550260,0.178301,0,0,13,0,13,13,21,1.000000,1,1,1,1,1,1,0.400000,0.000000,0.400000,0.0,0.755708,0.0,0.400000,0.0,0.755708,0.0,0.755708,0.0,1.126633,0.0,0.000000,0.000000,0.329548,0.000000,0.329548,0.329548,0.357081,0.464380,0,0.969604,10.498117,9.931219,0.0,0,0.0,4.242483,3.548925,0,83.506037,2976.754608,3811.021961,74.815098,42.47280,6905.064466,0,0,0,0.348256,0.000000,0.000000,0.348256,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.433984
1,8.237146,53.528878,12.975564,6.856623,2.709266,55.813529,8.027597,114.687272,0.178301,12,0,0,0,14,21,23,0.515883,1,1,1,1,1,1,0.375733,0.127767,0.400000,0.0,0.400000,0.0,0.400000,0.0,0.801944,0.0,1.126633,0.0,1.221084,0.0,0.318152,0.000000,0.000000,0.000000,0.334436,0.357081,0.360838,1.926852,0,0.341754,11.637102,11.008699,0.0,0,0.0,5.714286,4.384342,0,58.819068,3062.055300,3920.229085,74.764101,49.55160,7106.600086,0,0,0,0.245638,0.127767,0.348256,0.117871,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,2.268607
2,8.203394,43.348862,15.530304,6.494898,1.555715,48.910876,9.931528,115.948024,0.178301,17,0,0,0,13,20,20,0.515883,1,1,1,1,1,1,0.494341,0.168099,0.400000,0.0,0.400000,0.0,0.400000,0.0,0.755708,0.0,1.082228,0.0,1.082228,0.0,0.342575,0.000000,0.000000,0.000000,0.329548,0.354030,0.354030,0.249807,0,0.176353,8.249580,7.804102,0.0,0,0.0,8.099626,1.553322,0,72.384207,2990.199537,3828.234975,73.436949,49.55160,6941.423061,0,0,0,0.301554,0.168099,0.466127,0.133455,6.000000,0.0,6.500000,0.0,0.500000,4.000000,1.000000,0.426160
3,8.179529,43.443902,18.568457,6.311225,2.557439,42.233296,11.588044,117.205815,0.178301,15,0,0,0,13,13,19,0.515883,1,1,1,1,1,1,0.442914,0.150612,0.400000,0.0,0.400000,0.0,0.400000,0.0,0.755708,0.0,0.755708,0.0,1.038487,0.0,0.337368,0.000000,0.000000,0.000000,0.329548,0.329548,0.350495,0.257648,0,0.528722,9.722816,9.197784,0.0,0,0.0,7.142857,3.034631,0,72.228394,2716.945697,3478.398821,73.861596,42.47280,6311.678914,0,0,0,0.300577,0.150612,0.599582,0.149966,7.133645,0.0,6.128259,0.0,0.471405,3.559026,1.247219,0.786370
4,8.138330,31.389378,19.841654,6.076907,3.800219,43.376474,11.390929,116.765136,0.178301,18,9,0,0,13,14,21,0.515883,1,1,1,1,1,1,0.517627,0.176017,0.632133,0.0,0.400000,0.0,0.400000,0.0,0.755708,0.0,0.801944,0.0,1.126633,0.0,0.346408,0.272749,0.000000,0.000000,0.329548,0.334436,0.357081,0.283710,0,0.787248,12.559281,11.881080,0.0,0,0.0,16.980536,5.390377,0,82.213187,3002.861856,3844.446044,86.231003,53.09100,6986.629903,0,0,0,0.343307,0.176017,0.749548,0.167289,6.595453,0.0,5.629165,0.0,0.433013,3.766630,1.479020,1.070958
5,8.140649,31.267859,21.000000,6.843456,4.135672,44.630468,13

In [20]:
period = 1

simulation_per1, x_record_per1, error_record_per1 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per1, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  6493.696
Total cost: 6860.737
Penalty cost: 0.0
NOx emission cost: 472.243
H2 incentive: 377.041
Reserved H2 for the next round: 0.917
Deviations cost: 10
Operation cost: 41164.421
>>>>>>>>>>>>> The optimization is finished.


In [21]:
opt_solution_index_per1 = np.where(error_record_per1 == error_record_per1.min())
opt_solution_index_per1 = opt_solution_index_per1[0][0]
best_simulation_per1 = simulation_per1[opt_solution_index_per1]
best_itr_x_per1 = x_record_per1[opt_solution_index_per1]

display(best_itr_x_per1)

array([ 5.00000000e-01,  8.35060375e-01,  1.00000000e+00,  5.00000000e-01,
        5.88190679e-01, -2.41528138e-04,  5.00000000e-01,  7.23842070e-01,
       -2.41528138e-04,  5.00000000e-01,  7.22283940e-01, -2.41528138e-04,
        5.00000000e-01,  8.22131874e-01, -2.41528138e-04,  5.00000000e-01,
        8.76893458e-01, -2.41528138e-04])

In [22]:
period = 1

simulation_per1, x_record_per1, error_record_per1 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per1, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  6493.696
Total cost: 6860.737
Penalty cost: 0.0
NOx emission cost: 472.243
H2 incentive: 377.041
Reserved H2 for the next round: 0.917
Deviations cost: 10
Operation cost: 41164.421
>>>>>>>>>>>>> The optimization is finished.


In [23]:
opt_solution_index_per1 = np.where(error_record_per1 == error_record_per1.min())
opt_solution_index_per1 = opt_solution_index_per1[0][0]
best_simulation_per1 = simulation_per1[opt_solution_index_per1]
best_itr_x_per1 = x_record_per1[opt_solution_index_per1]

display(best_itr_x_per1)

array([ 5.00000000e-01,  8.35060375e-01,  1.00000000e+00,  5.00000000e-01,
        5.88190679e-01, -2.41528138e-04,  5.00000000e-01,  7.23842070e-01,
       -2.41528138e-04,  5.00000000e-01,  7.22283940e-01, -2.41528138e-04,
        5.00000000e-01,  8.22131874e-01, -2.41528138e-04,  5.00000000e-01,
        8.76893458e-01, -2.41528138e-04])

In [24]:
period = 1

simulation_per1, x_record_per1, error_record_per1 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per1, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  6493.696
Total cost: 6860.737
Penalty cost: 0.0
NOx emission cost: 472.243
H2 incentive: 377.041
Reserved H2 for the next round: 0.917
Deviations cost: 10
Operation cost: 41164.421
>>>>>>>>>>>>> The optimization is finished.


In [25]:
opt_solution_index_per1 = np.where(error_record_per1 == error_record_per1.min())
opt_solution_index_per1 = opt_solution_index_per1[0][0]
best_simulation_per1 = simulation_per1[opt_solution_index_per1]
best_itr_x_per1 = x_record_per1[opt_solution_index_per1]
print('Total cost:',best_simulation_per1['cost_total [EUR]'].sum())

display(best_itr_x_per1)


Total cost: 41164.421383421955


array([ 5.00000000e-01,  8.35060375e-01,  1.00000000e+00,  5.00000000e-01,
        5.88190679e-01, -2.41528138e-04,  5.00000000e-01,  7.23842070e-01,
       -2.41528138e-04,  5.00000000e-01,  7.22283940e-01, -2.41528138e-04,
        5.00000000e-01,  8.22131874e-01, -2.41528138e-04,  5.00000000e-01,
        8.76893458e-01, -2.41528138e-04])

In [26]:
period = 1

best_simulation_per1, cost_total_avg, penalty_total_avg =  MG_run_evaluation(df.iloc[(period-1)*hours:(period)*hours], best_itr_x_per1, parameter_definition, reserve_H2_op[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',best_simulation_per1['cost_total [EUR]'].sum())



Total cost: 40716.646242516785


In [27]:
for param in MG_run_cb_per1.columns:

    y1 = MG_run_cb_per1[param]
    y2 = best_simulation_per1[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [28]:
reserve_H2_cb[1] = MG_run_cb_per1['H2_tank_aval [kg/s]'].iloc[-1] + MG_run_cb_per1['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period condition-based:', parameter_formater(reserve_H2_cb[1],3))

reserve_H2_op[1] = best_simulation_per1['H2_tank_aval [kg/s]'].iloc[-1] + best_simulation_per1['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period optimization:', parameter_formater(reserve_H2_op[1],3))



H2 Tank content for next period condition-based: 1.343
H2 Tank content for next period optimization: 1.12


In [29]:
period = 2


MG_run_cb_per2, cost_total_cb_per2, penalty_total_cb_per2 = MG_run_cb(df.iloc[(period-1)*hours:(period)*hours], reserve_H2_cb[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',MG_run_cb_per2['cost_total [EUR]'].sum())
x_initial_from_cb = extract_x_span_for_initialization(MG_run_cb_per2, x_band_statements)    
simulation_per2, x_record_per2, error_record_per2 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, x_initial_from_cb, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='Nelder-Mead',sell=sell,buy=buy,display_step=display_step)





Total cost: 39070.41493568238


 
>>>>>> itr =  0  cost =  6032.846
Total cost: 6558.661
Penalty cost: 0.0
NOx emission cost: 455.549
H2 incentive: 525.815
Reserved H2 for the next round: 1.474
Deviations cost: 0
Operation cost: 39351.968
>>>>>>>>>>>>> The optimization is finished.


In [30]:
opt_solution_index_per2 = np.where(error_record_per2 == error_record_per2.min())
opt_solution_index_per2 = opt_solution_index_per2[0][0]
best_simulation_per2 = simulation_per2[opt_solution_index_per2]
best_itr_x_per2 = x_record_per2[opt_solution_index_per2]

display(best_itr_x_per2)

array([ 5.00000000e-01,  4.80734878e-01, -2.41528138e-04,  5.00000000e-01,
        4.72276555e-01, -2.41528138e-04,  5.00000000e-01,  5.09375172e-01,
       -2.41528138e-04,  5.00000000e-01,  6.42857544e-01, -2.41528138e-04,
        5.00000000e-01,  6.79959833e-01, -2.41528138e-04,  5.00000000e-01,
        5.47054481e-01, -2.41528138e-04])

In [31]:
period = 2

simulation_per2, x_record_per2, error_record_per2 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per2, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  6032.846
Total cost: 6558.661
Penalty cost: 0.0
NOx emission cost: 455.549
H2 incentive: 525.815
Reserved H2 for the next round: 1.474
Deviations cost: 0
Operation cost: 39351.968
>>>>>>>>>>>>> The optimization is finished.


In [32]:
opt_solution_index_per2 = np.where(error_record_per2 == error_record_per2.min())
opt_solution_index_per2 = opt_solution_index_per2[0][0]
best_simulation_per2 = simulation_per2[opt_solution_index_per2]
best_itr_x_per2 = x_record_per2[opt_solution_index_per2]

display(best_itr_x_per2)

array([ 5.00000000e-01,  4.80734878e-01, -2.41528138e-04,  5.00000000e-01,
        4.72276555e-01, -2.41528138e-04,  5.00000000e-01,  5.09375172e-01,
       -2.41528138e-04,  5.00000000e-01,  6.42857544e-01, -2.41528138e-04,
        5.00000000e-01,  6.79959833e-01, -2.41528138e-04,  5.00000000e-01,
        5.47054481e-01, -2.41528138e-04])

In [33]:
period = 2

simulation_per2, x_record_per2, error_record_per2 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per2, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  6032.846
Total cost: 6558.661
Penalty cost: 0.0
NOx emission cost: 455.549
H2 incentive: 525.815
Reserved H2 for the next round: 1.474
Deviations cost: 0
Operation cost: 39351.968
>>>>>>>>>>>>> The optimization is finished.


In [34]:
opt_solution_index_per2 = np.where(error_record_per2 == error_record_per2.min())
opt_solution_index_per2 = opt_solution_index_per2[0][0]
best_simulation_per2 = simulation_per2[opt_solution_index_per2]
best_itr_x_per2 = x_record_per2[opt_solution_index_per2]

display(best_itr_x_per2)

array([ 5.00000000e-01,  4.80734878e-01, -2.41528138e-04,  5.00000000e-01,
        4.72276555e-01, -2.41528138e-04,  5.00000000e-01,  5.09375172e-01,
       -2.41528138e-04,  5.00000000e-01,  6.42857544e-01, -2.41528138e-04,
        5.00000000e-01,  6.79959833e-01, -2.41528138e-04,  5.00000000e-01,
        5.47054481e-01, -2.41528138e-04])

In [35]:
period = 2

simulation_per2, x_record_per2, error_record_per2 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per2, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  6032.846
Total cost: 6558.661
Penalty cost: 0.0
NOx emission cost: 455.549
H2 incentive: 525.815
Reserved H2 for the next round: 1.474
Deviations cost: 0
Operation cost: 39351.968
>>>>>>>>>>>>> The optimization is finished.


In [36]:
opt_solution_index_per2 = np.where(error_record_per2 == error_record_per2.min())
opt_solution_index_per2 = opt_solution_index_per2[0][0]
best_simulation_per2 = simulation_per2[opt_solution_index_per2]
best_itr_x_per2 = x_record_per2[opt_solution_index_per2]
print('Total cost:',best_simulation_per2['cost_total [EUR]'].sum())
display(best_itr_x_per2)

Total cost: 39351.96829479841


array([ 5.00000000e-01,  4.80734878e-01, -2.41528138e-04,  5.00000000e-01,
        4.72276555e-01, -2.41528138e-04,  5.00000000e-01,  5.09375172e-01,
       -2.41528138e-04,  5.00000000e-01,  6.42857544e-01, -2.41528138e-04,
        5.00000000e-01,  6.79959833e-01, -2.41528138e-04,  5.00000000e-01,
        5.47054481e-01, -2.41528138e-04])

In [37]:
period = 2

best_simulation_per2, cost_total_avg, penalty_total_avg =  MG_run_evaluation(df.iloc[(period-1)*hours:(period)*hours], best_itr_x_per2, parameter_definition, reserve_H2_op[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',best_simulation_per2['cost_total [EUR]'].sum())



Total cost: 40582.61878393013


In [38]:
for param in MG_run_cb_per2.columns:

    y1 = MG_run_cb_per2[param]
    y2 = best_simulation_per2[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [39]:
reserve_H2_cb[2] = MG_run_cb_per2['H2_tank_aval [kg/s]'].iloc[-1] + MG_run_cb_per2['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period condition-based:', parameter_formater(reserve_H2_cb[2],3))

reserve_H2_op[2] = best_simulation_per2['H2_tank_aval [kg/s]'].iloc[-1] + best_simulation_per2['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period optimization:', parameter_formater(reserve_H2_op[2],3))



H2 Tank content for next period condition-based: 2.054
H2 Tank content for next period optimization: 1.546


In [40]:
period = 3


MG_run_cb_per3, cost_total_cb_per3, penalty_total_cb_per3 = MG_run_cb(df.iloc[(period-1)*hours:(period)*hours], reserve_H2_cb[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',MG_run_cb_per3['cost_total [EUR]'].sum())
x_initial_from_cb = extract_x_span_for_initialization(MG_run_cb_per3, x_band_statements)    
simulation_per3, x_record_per3, error_record_per3 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, x_initial_from_cb, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='Nelder-Mead',sell=sell,buy=buy,display_step=display_step)





Total cost: 36610.139352629965


 
>>>>>> itr =  0  cost =  21461.603
Total cost: 5580.658
Penalty cost: 16666.667
NOx emission cost: 401.385
H2 incentive: 785.721
Reserved H2 for the next round: 2.326
Deviations cost: 0
Operation cost: 33483.946
>>>>>>>>>>>>> The optimization is finished.


In [41]:
opt_solution_index_per3 = np.where(error_record_per3 == error_record_per3.min())
opt_solution_index_per3 = opt_solution_index_per3[0][0]
best_simulation_per3 = simulation_per3[opt_solution_index_per3]
best_itr_x_per3 = x_record_per3[opt_solution_index_per3]

display(best_itr_x_per3)

array([ 5.00000000e-01,  6.95067433e-01, -2.41528138e-04,  5.00000000e-01,
        4.68270041e-01, -2.41528138e-04,  5.00000000e-01,  7.05785208e-01,
       -2.41528138e-04,  5.00000000e-01,  6.27991601e-01, -2.41528138e-04,
        5.00000000e-01,  5.74005525e-01, -2.41528138e-04,  5.00000000e-01,
        4.01749832e-01, -2.41528138e-04])

In [42]:
period = 3

simulation_per3, x_record_per3, error_record_per3 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per3, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  21461.603
Total cost: 5580.658
Penalty cost: 16666.667
NOx emission cost: 401.385
H2 incentive: 785.721
Reserved H2 for the next round: 2.326
Deviations cost: 0
Operation cost: 33483.946
>>>>>>>>>>>>> The optimization is finished.


In [43]:
opt_solution_index_per3 = np.where(error_record_per3 == error_record_per3.min())
opt_solution_index_per3 = opt_solution_index_per3[0][0]
best_simulation_per3 = simulation_per3[opt_solution_index_per3]
best_itr_x_per3 = x_record_per3[opt_solution_index_per3]

display(best_itr_x_per3)

array([ 5.00000000e-01,  6.95067433e-01, -2.41528138e-04,  5.00000000e-01,
        4.68270041e-01, -2.41528138e-04,  5.00000000e-01,  7.05785208e-01,
       -2.41528138e-04,  5.00000000e-01,  6.27991601e-01, -2.41528138e-04,
        5.00000000e-01,  5.74005525e-01, -2.41528138e-04,  5.00000000e-01,
        4.01749832e-01, -2.41528138e-04])

In [44]:
period = 3

simulation_per3, x_record_per3, error_record_per3 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per3, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  21461.603
Total cost: 5580.658
Penalty cost: 16666.667
NOx emission cost: 401.385
H2 incentive: 785.721
Reserved H2 for the next round: 2.326
Deviations cost: 0
Operation cost: 33483.946
>>>>>>>>>>>>> The optimization is finished.


In [45]:
opt_solution_index_per3 = np.where(error_record_per3 == error_record_per3.min())
opt_solution_index_per3 = opt_solution_index_per3[0][0]
best_simulation_per3 = simulation_per3[opt_solution_index_per3]
best_itr_x_per3 = x_record_per3[opt_solution_index_per3]

display(best_itr_x_per3)

array([ 5.00000000e-01,  6.95067433e-01, -2.41528138e-04,  5.00000000e-01,
        4.68270041e-01, -2.41528138e-04,  5.00000000e-01,  7.05785208e-01,
       -2.41528138e-04,  5.00000000e-01,  6.27991601e-01, -2.41528138e-04,
        5.00000000e-01,  5.74005525e-01, -2.41528138e-04,  5.00000000e-01,
        4.01749832e-01, -2.41528138e-04])

In [46]:
period = 3

simulation_per3, x_record_per3, error_record_per3 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per3, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  21461.603
Total cost: 5580.658
Penalty cost: 16666.667
NOx emission cost: 401.385
H2 incentive: 785.721
Reserved H2 for the next round: 2.326
Deviations cost: 0
Operation cost: 33483.946
>>>>>>>>>>>>> The optimization is finished.


In [47]:
opt_solution_index_per3 = np.where(error_record_per3 == error_record_per3.min())
opt_solution_index_per3 = opt_solution_index_per3[0][0]
best_simulation_per3 = simulation_per3[opt_solution_index_per3]
best_itr_x_per3 = x_record_per3[opt_solution_index_per3]
print('Total cost:',best_simulation_per3['cost_total [EUR]'].sum())

display(best_itr_x_per3)

Total cost: 33483.9459285369


array([ 5.00000000e-01,  6.95067433e-01, -2.41528138e-04,  5.00000000e-01,
        4.68270041e-01, -2.41528138e-04,  5.00000000e-01,  7.05785208e-01,
       -2.41528138e-04,  5.00000000e-01,  6.27991601e-01, -2.41528138e-04,
        5.00000000e-01,  5.74005525e-01, -2.41528138e-04,  5.00000000e-01,
        4.01749832e-01, -2.41528138e-04])

In [48]:
period = 3

best_simulation_per3, cost_total_avg, penalty_total_avg =  MG_run_evaluation(df.iloc[(period-1)*hours:(period)*hours], best_itr_x_per3, parameter_definition, reserve_H2_op[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',best_simulation_per3['cost_total [EUR]'].sum())



Total cost: 35585.706893132956


In [49]:
for param in MG_run_cb_per3.columns:

    y1 = MG_run_cb_per3[param]
    y2 = best_simulation_per3[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [50]:
reserve_H2_cb[3] = MG_run_cb_per3['H2_tank_aval [kg/s]'].iloc[-1] + MG_run_cb_per3['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period condition-based:', parameter_formater(reserve_H2_cb[3],3))

reserve_H2_op[3] = best_simulation_per3['H2_tank_aval [kg/s]'].iloc[-1] + best_simulation_per3['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period optimization:', parameter_formater(reserve_H2_op[3],3))



H2 Tank content for next period condition-based: 2.827
H2 Tank content for next period optimization: 2.288


In [51]:
period = 4


MG_run_cb_per4, cost_total_cb_per4, penalty_total_cb_per4 = MG_run_cb(df.iloc[(period-1)*hours:(period)*hours], reserve_H2_cb[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',MG_run_cb_per4['cost_total [EUR]'].sum())
x_initial_from_cb = extract_x_span_for_initialization(MG_run_cb_per4, x_band_statements)    
simulation_per4, x_record_per4, error_record_per4 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, x_initial_from_cb, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='Nelder-Mead',sell=sell,buy=buy,display_step=display_step)





Total cost: 40531.81272596917


 
>>>>>> itr =  0  cost =  22232.928
Total cost: 6117.945
Penalty cost: 16666.667
NOx emission cost: 431.085
H2 incentive: 581.684
Reserved H2 for the next round: 1.961
Deviations cost: 30
Operation cost: 36707.673
>>>>>>>>>>>>> The optimization is finished.


In [52]:
opt_solution_index_per4 = np.where(error_record_per4 == error_record_per4.min())
opt_solution_index_per4 = opt_solution_index_per4[0][0]
best_simulation_per4 = simulation_per4[opt_solution_index_per4]
best_itr_x_per4 = x_record_per4[opt_solution_index_per4]

display(best_itr_x_per4)

array([ 5.00000000e-01,  3.50915814e-01, -2.41528138e-04,  5.00000000e-01,
        2.03732896e-01, -2.41528138e-04,  5.00000000e-01,  2.46947355e-01,
       -2.41528138e-04,  5.00000000e-01,  0.00000000e+00, -2.41528138e-04,
        5.00000000e-01,  0.00000000e+00, -2.41528138e-04,  5.00000000e-01,
        0.00000000e+00, -2.41528138e-04])

In [53]:
period = 4

simulation_per4, x_record_per4, error_record_per4 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per4, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  22232.928
Total cost: 6117.945
Penalty cost: 16666.667
NOx emission cost: 431.085
H2 incentive: 581.684
Reserved H2 for the next round: 1.961
Deviations cost: 30
Operation cost: 36707.673
>>>>>>>>>>>>> The optimization is finished.


In [54]:
opt_solution_index_per4 = np.where(error_record_per4 == error_record_per4.min())
opt_solution_index_per4 = opt_solution_index_per4[0][0]
best_simulation_per4 = simulation_per4[opt_solution_index_per4]
best_itr_x_per4 = x_record_per4[opt_solution_index_per4]

display(best_itr_x_per4)

array([ 5.00000000e-01,  3.50915814e-01, -2.41528138e-04,  5.00000000e-01,
        2.03732896e-01, -2.41528138e-04,  5.00000000e-01,  2.46947355e-01,
       -2.41528138e-04,  5.00000000e-01,  0.00000000e+00, -2.41528138e-04,
        5.00000000e-01,  0.00000000e+00, -2.41528138e-04,  5.00000000e-01,
        0.00000000e+00, -2.41528138e-04])

In [55]:
period = 4

simulation_per4, x_record_per4, error_record_per4 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per4, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  22232.928
Total cost: 6117.945
Penalty cost: 16666.667
NOx emission cost: 431.085
H2 incentive: 581.684
Reserved H2 for the next round: 1.961
Deviations cost: 30
Operation cost: 36707.673
>>>>>>>>>>>>> The optimization is finished.


In [56]:
opt_solution_index_per4 = np.where(error_record_per4 == error_record_per4.min())
opt_solution_index_per4 = opt_solution_index_per4[0][0]
best_simulation_per4 = simulation_per4[opt_solution_index_per4]
best_itr_x_per4 = x_record_per4[opt_solution_index_per4]

display(best_itr_x_per4)

array([ 5.00000000e-01,  3.50915814e-01, -2.41528138e-04,  5.00000000e-01,
        2.03732896e-01, -2.41528138e-04,  5.00000000e-01,  2.46947355e-01,
       -2.41528138e-04,  5.00000000e-01,  0.00000000e+00, -2.41528138e-04,
        5.00000000e-01,  0.00000000e+00, -2.41528138e-04,  5.00000000e-01,
        0.00000000e+00, -2.41528138e-04])

In [57]:
period = 4

simulation_per4, x_record_per4, error_record_per4 = optimize_microgrid_performance(df.iloc[(period-1)*hours:(period)*hours], parameter_definition, best_itr_x_per4, x_band_statements, x_band_statements, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='ga',sell=sell,buy=buy,display_step=display_step)



 
>>>>>> itr =  0  cost =  22232.928
Total cost: 6117.945
Penalty cost: 16666.667
NOx emission cost: 431.085
H2 incentive: 581.684
Reserved H2 for the next round: 1.961
Deviations cost: 30
Operation cost: 36707.673
>>>>>>>>>>>>> The optimization is finished.


In [58]:
opt_solution_index_per4 = np.where(error_record_per4 == error_record_per4.min())
opt_solution_index_per4 = opt_solution_index_per4[0][0]
best_simulation_per4 = simulation_per4[opt_solution_index_per4]
best_itr_x_per4 = x_record_per4[opt_solution_index_per4]
print('Total cost:',best_simulation_per4['cost_total [EUR]'].sum())

display(best_itr_x_per4)

Total cost: 36707.6728589805


array([ 5.00000000e-01,  3.50915814e-01, -2.41528138e-04,  5.00000000e-01,
        2.03732896e-01, -2.41528138e-04,  5.00000000e-01,  2.46947355e-01,
       -2.41528138e-04,  5.00000000e-01,  0.00000000e+00, -2.41528138e-04,
        5.00000000e-01,  0.00000000e+00, -2.41528138e-04,  5.00000000e-01,
        0.00000000e+00, -2.41528138e-04])

In [59]:
 period = 4

best_simulation_per4, cost_total_avg, penalty_total_avg =  MG_run_evaluation(df.iloc[(period-1)*hours:(period)*hours], best_itr_x_per4, parameter_definition, reserve_H2_op[period-1], x_band_statements, buy=buy, sell=sell)
print('Total cost:',best_simulation_per4['cost_total [EUR]'].sum())



Total cost: 38722.89849816716


In [60]:
for param in MG_run_cb_per4.columns:

    y1 = MG_run_cb_per4[param]
    y2 = best_simulation_per4[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [61]:
reserve_H2_cb[4] = MG_run_cb_per4['H2_tank_aval [kg/s]'].iloc[-1] + MG_run_cb_per4['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period condition-based:', parameter_formater(reserve_H2_cb[4],3))

reserve_H2_op[4] = best_simulation_per4['H2_tank_aval [kg/s]'].iloc[-1] + best_simulation_per4['H2_tank_var [kg/s]'].iloc[-1]
print('H2 Tank content for next period optimization:', parameter_formater(reserve_H2_op[4],3))



H2 Tank content for next period condition-based: 2.31
H2 Tank content for next period optimization: 1.704


In [62]:
best_simulation_day1 = pd.concat((best_simulation_per1,best_simulation_per2,best_simulation_per3,best_simulation_per4),axis=0).reset_index(drop=True)
MG_run_cb_day1 = pd.concat((MG_run_cb_per1,MG_run_cb_per2,MG_run_cb_per3,MG_run_cb_per4),axis=0).reset_index(drop=True)




In [63]:
for param in MG_run_cb_day1.columns:

    y1 = MG_run_cb_day1[param]
    y2 = best_simulation_day1[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [64]:
# optimization parameters with [min, max, initial guess]


opt_param_dict_daily = {
'P_GFA_to_GFC':[-20,20,10],
'P_GFA_to_ELZ':[0,100,0],
'FR_GFA_GT1':[0.516,1,1],
# 'FR_GFA_GT2':[0,1,1],
# 'FR_GFA_GT3':[0,1,1],
# 'FR_GFA_GT4':[0.516,1,1],
}

x_band_daily, x_band_statements_daily, parameter_definition_daily, x_initial_daily, no_initial_pop_daily, initial_pop_daily = MG_parameter_boundary_def(opt_param_dict_daily, observation_window_daily)



In [65]:
day = 1
period = 1

MG_run_cb_day1, cost_total_cb_day1, penalty_total_cb_day1 = MG_run_cb(df.iloc[(day-1)*hours:(day)*day_hours], reserve_H2_cb[period-1], x_band_statements_daily, buy=buy, sell=sell)
print('Total cost:',MG_run_cb_day1['cost_total [EUR]'].sum())

x_initial_from_day1 = extract_x_span_for_initialization(best_simulation_day1, x_band_statements_daily)    
simulation_day1, x_record_day1, error_record_day1 = optimize_microgrid_performance(df.iloc[(day-1)*hours:(day)*day_hours], parameter_definition_daily, x_initial_from_day1, x_band_statements, x_band_statements_daily, initial_pop,reserve_H2_op[period-1],\
                                                                                              display=False,actual_band=x_band,method='Nelder-Mead',sell=sell,buy=buy,display_step=display_step)





Total cost: 156344.95968923427


 
>>>>>> itr =  0  cost =  14467.586
Total cost: 6279.5
Penalty cost: 8333.333
NOx emission cost: 1760.263
H2 incentive: 155.248
Reserved H2 for the next round: 2.079
Deviations cost: 10
Operation cost: 150708.008
>>>>>>>>>>>>> The optimization is finished.


In [66]:
opt_solution_index_day1 = np.where(error_record_day1 == error_record_day1.min())
opt_solution_index_day1 = opt_solution_index_day1[0][0]
best_simulation_day1 = simulation_day1[opt_solution_index_day1]
best_itr_x_day1 = x_record_day1[opt_solution_index_day1]
print('Total cost:',best_simulation_day1['cost_total [EUR]'].sum())

display(best_itr_x_day1)

Total cost: 150708.00846573777


array([ 5.00000000e-01,  8.35060375e-01,  1.00000000e+00,  5.00000000e-01,
        5.88190679e-01, -2.41528138e-04,  5.00000000e-01,  7.23842070e-01,
       -2.41528138e-04,  5.00000000e-01,  7.22283940e-01, -2.41528138e-04,
        5.00000000e-01,  8.22131874e-01, -2.41528138e-04,  5.00000000e-01,
        8.76893458e-01, -2.41528138e-04,  5.00000000e-01,  4.80734878e-01,
       -2.41528138e-04,  5.00000000e-01,  4.72276555e-01, -2.41528138e-04,
        5.00000000e-01,  5.09375172e-01, -2.41528138e-04,  5.00000000e-01,
        6.42857544e-01, -2.41528138e-04,  5.00000000e-01,  6.79959833e-01,
       -2.41528138e-04,  5.00000000e-01,  5.47054481e-01, -2.41528138e-04,
        5.00000000e-01,  6.95067433e-01, -2.41528138e-04,  5.00000000e-01,
        4.68270041e-01, -2.41528138e-04,  5.00000000e-01,  7.05785208e-01,
       -2.41528138e-04,  5.00000000e-01,  6.27991601e-01, -2.41528138e-04,
        5.00000000e-01,  5.74005525e-01, -2.41528138e-04,  5.00000000e-01,
        4.01749832e-01, -

In [67]:
print('H2 Tank content for next period condition-based:', parameter_formater(MG_run_cb_day1['H2_tank_aval [kg/s]'].iloc[-1]+MG_run_cb_day1['H2_tank_var [kg/s]'].iloc[-1],3))

print('H2 Tank content for next period optimization:', parameter_formater(best_simulation_day1['H2_tank_aval [kg/s]'].iloc[-1]+best_simulation_day1['H2_tank_var [kg/s]'].iloc[-1],3))


H2 Tank content for next period condition-based: 2.31
H2 Tank content for next period optimization: 1.866


In [68]:
day = 1
period = 1

best_simulation_day1, cost_total_avg, penalty_total_avg =  MG_run_evaluation(df.iloc[(day-1)*hours:(day)*day_hours], best_itr_x_day1, parameter_definition_daily, reserve_H2_op[period-1], x_band_statements_daily, buy=buy, sell=sell)
print('Total cost:',best_simulation_day1['cost_total [EUR]'].sum())



Total cost: 155607.870417747


In [69]:
for param in MG_run_cb_day1.columns:

    y1 = MG_run_cb_day1[param]
    y2 = best_simulation_day1[param]
    x_plot = np.arange(0,len(y1))
    x_show = np.linspace(0,len(y1),num=6)

    leg = ['condition-based','optimized']
    simple_plotter(x_plot,[y1,y2],display_legend=leg,x_show=x_show,x_label='Time [hour]',y_label=param,shape=[14,5])


In [70]:
save_df_excel_to_path(MG_run_cb_day1,'performance_day 1','condition-based',path_saving_data_results)
save_df_excel_to_path(best_simulation_day1,'performance_day 1','optimized',path_saving_data_results)


The data-frame is saved in: /Users/2923185/Desktop/Offshore Microgrid clean/Codes/outputs/results/Offshore Microgrid Optimization - IM - GF - Day 1/performance_day 1.xlsx
The data-frame is saved in: /Users/2923185/Desktop/Offshore Microgrid clean/Codes/outputs/results/Offshore Microgrid Optimization - IM - GF - Day 1/performance_day 1.xlsx
